In [ ]:
# Libraries to help with reading and manipulating data
import pandas as pd
import numpy as np
from datetime import datetime

# To suppress scientific notations
pd.set_option("display.float_format", lambda x: "%.3f" % x)

# Libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# To tune model, get different metric scores, and split data
from sklearn import metrics
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# To be used for data scaling and one hot encoding
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder

# To impute missing values
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer

# To oversample and undersample data
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# To do hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

# To define maximum number of columns to be displayed in a dataframe
pd.set_option("display.max_columns", None)

# To supress scientific notations for a dataframe
pd.set_option("display.float_format", lambda x: "%.3f" % x)

# To help with model building
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    BaggingClassifier,
)
from xgboost import XGBClassifier

# To supress warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
Traveldata_train = pd.read_csv('Traveldata_train.csv')
Traveldata_test = pd.read_csv('Traveldata_test.csv')

Surveydata_train = pd.read_csv('Surveydata_train.csv')
Surveydata_test = pd.read_csv('Surveydata_test.csv')


In [ ]:
Data = pd.merge(Traveldata_train, Surveydata_train, on="ID", how="inner")


In [ ]:
Test_data = pd.merge(Traveldata_test, Surveydata_test, on="ID", how="inner")


In [ ]:
# copying data to another variable to avoid any changes to original data
data = Data.copy()

In [ ]:
Tdata = Test_data.copy()

In [ ]:
Traveldata_train.head()

In [ ]:
Traveldata_train.shape

In [ ]:
Surveydata_train.shape

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
Tdata.info()

In [ ]:
# checking for null values
data.isnull().sum()

In [ ]:
# checking for null values
Tdata.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.head()

In [ ]:
data.Overall_Experience.nunique()

In [ ]:
for column in data.columns:
    missing_percentage = data[column].isnull().sum() / data.shape[0] * 100
    print(f"{column}: {missing_percentage:.2f}% missing")
    print("-" * 50)

In [ ]:
numerical_cols = data.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = data.select_dtypes(include=["object"]).columns.tolist()


In [ ]:
cols_list = numerical_cols + categorical_cols

cols_list

In [ ]:
# Impute numerical columns with median
  # Replace with relevant numerical column names
#median_imputer = SimpleImputer(strategy="median")
#data[numerical_cols] = median_imputer.fit_transform(data[numerical_cols])

In [ ]:
# Impute categorical columns with mode
#mode_imputer = SimpleImputer(strategy="most_frequent")
#data[categorical_cols] = mode_imputer.fit_transform(data[categorical_cols])

In [ ]:
Tnumerical_cols = Tdata.select_dtypes(include=["number"]).columns.tolist()
Tcategorical_cols = Tdata.select_dtypes(include=["object"]).columns.tolist()


In [ ]:
Tcols_list = Tnumerical_cols + Tcategorical_cols

Tcols_list

In [ ]:
# Impute numerical columns with median
  #Replace with relevant numerical column names
#Tmedian_imputer = SimpleImputer(strategy="median")
#Tdata[Tnumerical_cols] = Tmedian_imputer.fit_transform(Tdata[Tnumerical_cols])

In [ ]:
# Impute categorical columns with mode
#Tmode_imputer = SimpleImputer(strategy="most_frequent")
#Tdata[Tcategorical_cols] = Tmode_imputer.fit_transform(Tdata[Tcategorical_cols])

In [ ]:
# Check for missing values after dropping the column
#Tmissing_values = (Tdata.isnull().sum() / Tdata.shape[0]) * 100

In [ ]:
# checking for null values
#Tmissing_values

In [ ]:
Tdata

In [ ]:
# Define the mapping for ordinal columns
ordinal_mapping = {
    "Extremely Poor": 1,
    "Poor": 2,
    "Needs Improvement": 3,
    "Acceptable": 4,
    "Good": 5,
    "Excellent": 6
}

# List of columns to encode
ordinal_columns = [
    "Arrival_Time_Convenient","Seat_Comfort", "Catering", "Onboard_Wifi_Service",
    "Onboard_Entertainment", "Online_Support", "Ease_of_Online_Booking",
    "Onboard_Service", "Legroom", "Baggage_Handling", "CheckIn_Service",
    "Cleanliness", "Online_Boarding"
]

# Apply the mapping to encode ordinal columns
for column in ordinal_columns:
    data[column] = data[column].map(ordinal_mapping)


In [ ]:
# Define the mapping for Platform_Location
platform_location_mapping = {
    "Very Inconvenient": 1,
    "Inconvenient": 2,
    "Needs Improvement": 3,
    "Manageable": 4,
    "Convenient": 5,
    "Very Convenient": 6
}

# Apply the mapping to encode Platform_Location
data["Platform_Location"] = data["Platform_Location"].map(platform_location_mapping)

In [ ]:
data.head()

In [ ]:
# Apply the mapping to encode ordinal columns
for column in ordinal_columns:
    Tdata[column] = Tdata[column].map(ordinal_mapping)

In [ ]:
# Apply the mapping to encode Platform_Location
Tdata["Platform_Location"] = Tdata["Platform_Location"].map(platform_location_mapping)

In [ ]:
data = data.drop(["ID"], axis=1)

In [ ]:
# defining X and y variables
X = data.drop(["Overall_Experience"], axis=1)
y = data["Overall_Experience"]

print(X.head())
print(y.head())

In [ ]:
# creating dummy variables
X = pd.get_dummies(
    X,
    columns=X.select_dtypes(include=["object", "category"]).columns.tolist(),
    drop_first=True
)
X = X.astype(float)
X.head()

In [ ]:
# Example: Assuming 'data' is your dataset
imputer = KNNImputer(n_neighbors=5, weights="distance")

# Applying KNN imputation
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

In [ ]:
# Check for missing values after dropping the column
missing_values = (X.isnull().sum() / X.shape[0]) * 100

In [ ]:
# checking for null values
missing_values

In [ ]:
# creating dummy variables 
X_Test = pd.get_dummies(
    Tdata,
    columns=Tdata.select_dtypes(include=["object", "category"]).columns.tolist(),
    drop_first=True
)
X_Test = X_Test.astype(float)
X_Test.head()

In [ ]:
# Applying KNN imputation
X_Test = pd.DataFrame(imputer.fit_transform(X_Test), columns=X_Test.columns)

In [ ]:
# Check for missing values after dropping the column
Tmissing_values = (X_Test.isnull().sum() / X_Test.shape[0]) * 100

In [ ]:
# checking for null values
Tmissing_values

In [ ]:


# then we split the temporary set into train and validation

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=1, stratify=y
)
print(X_train.shape, X_val.shape, X_Test.shape)

# Model Building

In [ ]:
# defining a function to compute different metrics to check performance of a classification model built using sklearn
def model_performance_classification_sklearn(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    acc = accuracy_score(target, pred)  # to compute Accuracy
    recall = recall_score(target, pred)  # to compute Recall
    precision = precision_score(target, pred)  # to compute Precision
    f1 = f1_score(target, pred)  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1,},
        index=[0],
    )

    return df_perf

In [ ]:
def confusion_matrix_sklearn(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    y_pred = model.predict(predictors)
    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

## Initial Model Building

### Model Building - Original Data

In [ ]:
models = []  # Empty list to store all the models

# Appending models into the list
models.append(("Bagging", BaggingClassifier(DecisionTreeClassifier(random_state=1, class_weight='balanced'), random_state=1)))
models.append(("Random forest", RandomForestClassifier(random_state=1, class_weight='balanced')))
models.append(("GBM", GradientBoostingClassifier(random_state=1)))
models.append(("Adaboost", AdaBoostClassifier(random_state=1)))
models.append(("dtree", DecisionTreeClassifier(random_state=1, class_weight='balanced')))

print("\nTraining Performance:\n")
for name, model in models:
    model.fit(X_train, y_train)
    scores = recall_score(y_train, model.predict(X_train))
    print("{}: {}".format(name, scores))

print("\nValidation Performance:\n")
for name, model in models:
    model.fit(X_train, y_train)
    scores_val = recall_score(y_val, model.predict(X_val))
    print("{}: {}".format(name, scores_val))

In [ ]:
print("\nTraining and Validation Performance Difference:\n")

for name, model in models:
    model.fit(X_train, y_train)
    scores_train = recall_score(y_train, model.predict(X_train))
    scores_val = recall_score(y_val, model.predict(X_val))
    difference1 = scores_train - scores_val
    print("{}: Training Score: {:.4f}, Validation Score: {:.4f}, Difference: {:.4f}".format(name, scores_train, scores_val, difference1))

- Adaboost has the best performance followed by GBM model as per the validation performance

## Hyperparameter Tuning

### Tuning  Gradient Boosting model with Original Data

In [ ]:
%%time

# Creating pipeline
Model = GradientBoostingClassifier(random_state=1)

# Parameter grid for RandomizedSearchCV
param_grid = {
    "init": [
        AdaBoostClassifier(random_state=1),
        DecisionTreeClassifier(random_state=1)
    ],
    "n_estimators": np.arange(125, 175, 10),  # Testing values from 125 to 165 in steps of 10
    "learning_rate": [0.01, 0.2, 0.05, 1],  # Explore different learning rates
    "subsample": [0.8, 0.9, 1],  # Fraction of samples used for training
    "max_features": [0.5, 0.7, 1],  # Fraction of features used for training
    "max_depth": [8,9, 10]  # Controlling the tree depth
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

# Configuring RandomizedSearchCV
randomized_cv = RandomizedSearchCV(
    estimator=Model,
    param_distributions=param_grid,
    n_iter=30,  # Randomly testing 20 combinations
    scoring=scorer,
    cv=3,  # 3-fold cross-validation
    random_state=1,
    n_jobs=-1  # Utilize all available CPU cores
)

# Fitting the RandomizedSearchCV to the training data
randomized_cv.fit(X_train, y_train)

# Outputting the best parameters and CV score
print("Best parameters are {} with CV score={}:" .format(
    randomized_cv.best_params_,
    randomized_cv.best_score_
))

In [ ]:
tuned_gbm2 = GradientBoostingClassifier(
    random_state=1,
    subsample=1,
    n_estimators=135,
    max_features=0.7,
    max_depth=9,
    learning_rate=0.2,
    init=AdaBoostClassifier(random_state=1),
)
tuned_gbm2.fit(X_train, y_train)

In [ ]:
# Checking model's performance on training set
gbm1_train = model_performance_classification_sklearn(
    tuned_gbm2, X_train, y_train)
gbm1_train

In [ ]:
# Checking model's performance on validation set
gbm1_val = model_performance_classification_sklearn(tuned_gbm2, X_val, y_val)
gbm1_val

### Feature Importance

In [ ]:
feature_names = X_train.columns
importances = tuned_gbm2.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12, 12))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

In [ ]:
print("Trained Features:", tuned_gbm2.feature_names_in_)

In [ ]:
# Reorder test data columns to match trained feature names
test_data = X_Test[tuned_gbm2.feature_names_in_]

In [ ]:
# Predict the target variable using the trained model

test_data["Overall_Experience"] = tuned_gbm2.predict(test_data)

In [ ]:
id_column = X_Test["ID"]

In [ ]:
test_data["ID"] = id_column

In [ ]:
test_data.to_csv('tuned_gbm2.csv', index=False)